In [ ]:
# Importance de l'embedding multilingue 

In [ ]:
from pymilvus import MilvusClient
import json
from tqdm import tqdm

from pymilvus import (
    connections, FieldSchema, CollectionSchema, DataType,
    Collection, utility
)
import ollama
from ollama import Client

In [48]:
import pymilvus
print(pymilvus.__version__)

2.6.5


# Index into milvus 

In [22]:
milvus_client = MilvusClient(
    uri="http://localhost:19530",
    token="root:Milvus"
)

collection_name = "rag_minist_int"

if milvus_client.has_collection(collection_name):
    milvus_client.drop_collection(collection_name)

In [ ]:
# milvus_client.create_collection(
#     collection_name=collection_name,
#     dimension=embedding_dim,
#     metric_type="IP",  # Inner product distance
#     consistency_level="Strong",  # Strong consistency level
#     replica_number=1,
# )

# # création de réplique de la BDD
# client.create_database(
#     db_name="my_database_2",
#     properties={
#         "database.replica.number": 1
#     }
# )





In [43]:
if client.has_collection("rag_minist_int_v2"):
    client.drop_collection("rag_minist_int_v2")

In [ ]:
import json
from tqdm import tqdm
from pymilvus import (
    MilvusClient, FieldSchema, CollectionSchema, DataType,
    Collection, connections, utility
)
from ollama import Client as OllamaClient

# -----------------------
# Config
# -----------------------
MILVUS_URI = "http://localhost:19530"
COLLECTION_NAME = "rag_minist_int_v2"   # nouveau nom
EMBED_DIM = 1024
BATCH_SIZE = 128
JSON_PATH = "chunks.json"
MAX_TEXT = 60000

# -----------------------
# Embeddings (Ollama)
# -----------------------
ollama_client = OllamaClient(host="http://localhost:11434")

def emb_text(text: str) -> list[float]:
    return ollama_client.embeddings(model="qwen3-embedding:0.6b", prompt=text)["embedding"]

# -----------------------
# Connexion (pour Collection API)
# -----------------------
connections.connect(alias="default", uri=MILVUS_URI)

# Client (insert pratique)
client = MilvusClient(uri=MILVUS_URI)

# -----------------------
# Create collection if needed
# -----------------------
if not utility.has_collection(COLLECTION_NAME):
    fields = [
        FieldSchema(name="id", dtype=DataType.VARCHAR, is_primary=True, max_length=512),
        FieldSchema(name="vector", dtype=DataType.FLOAT_VECTOR, dim=EMBED_DIM),
        FieldSchema(name="text", dtype=DataType.VARCHAR, max_length=65535),
        FieldSchema(name="source", dtype=DataType.VARCHAR, max_length=1024),
        FieldSchema(name="section_path", dtype=DataType.VARCHAR, max_length=512),
        FieldSchema(name="section_title", dtype=DataType.VARCHAR, max_length=512),
        FieldSchema(name="page_no", dtype=DataType.INT64),
        FieldSchema(name="chunk_type", dtype=DataType.VARCHAR, max_length=64),
    ]
    schema = CollectionSchema(
        fields,
        description="PDF chunks with embeddings",
        consistency_level="Strong",)

    # Crée la collection via Collection API (stable)
    col = Collection(name=COLLECTION_NAME, schema=schema)

    # Crée l'index via dict (stable)
    col.create_index(
        field_name="vector",
        index_params={
            "index_type": "HNSW",   # pas utilisable si embedding en BF18 ou 8B
            "metric_type": "COSINE",
            "params": {"M": 16, "efConstruction": 200}, # M = nb de voisin max par noeud dans le graph, efConstruction = taille du buffer de candidats explorés pendant la construction de l'index quand on insère chaque point
        },
    )

else:
    col = Collection(COLLECTION_NAME)

col.load()

# 1. Comment gérer les autorisations pour une base de données ?


In [5]:
client.create_user(user_name="user_1", password="P@ssw0rd")

In [6]:
client.list_users()

['root', 'user_1']

## 1. Simple recherche

In [45]:


# -----------------------
# Load JSON
# -----------------------
with open(JSON_PATH, "r", encoding="utf-8") as f:
    items = json.load(f)

# -----------------------
# Insert in batches (via MilvusClient)
# -----------------------
batch = []
inserted = 0

for obj in tqdm(items, desc="Embedding + insert"):
    text = (obj.get("text") or "")[:MAX_TEXT]
    meta = obj.get("meta") or {}

    vec = emb_text(text)
    if len(vec) != EMBED_DIM:
        raise ValueError(f"Bad embedding dim for id={obj.get('id')}: {len(vec)} != {EMBED_DIM}")

    batch.append({
        "id": obj["id"],
        "vector": vec,
        "text": text,
        "source": obj.get("source", ""),
        "section_path": obj.get("section_path", ""),
        "section_title": obj.get("section_title", ""),
        "page_no": int(obj.get("page_no", -1)) if obj.get("page_no") is not None else -1,
        "chunk_type": meta.get("type", ""),
    })

    if len(batch) >= BATCH_SIZE:
        client.insert(collection_name=COLLECTION_NAME, data=batch)
        inserted += len(batch)
        batch.clear()

if batch:
    client.insert(collection_name=COLLECTION_NAME, data=batch)
    inserted += len(batch)

print(f"Inserted {inserted} rows into {COLLECTION_NAME}")


Embedding + insert: 100%|██████████| 37/37 [00:16<00:00,  2.21it/s]


Inserted 37 rows into rag_minist_int_v2


In [47]:
def hit_score(hit):
    # pymilvus expose souvent .distance (même si c'est une similarité selon métrique)
    if hasattr(hit, "distance"):
        return hit.distance
    if hasattr(hit, "score"):
        return hit.score
    # dict fallback
    if isinstance(hit, dict):
        return hit.get("score", hit.get("distance"))
    return None

def hit_entity(hit):
    if hasattr(hit, "entity"):
        # parfois entity est dict-like
        return hit.entity
    if isinstance(hit, dict):
        return hit.get("entity", hit)
    return {}

question = "Quelle est l'idée principale de DeepSeek-V2 ?"

qvec = emb_text(question)
# si ton index est en IP (recommandé), normalise :
# qvec = l2_normalize(qvec)
    

res = client.search(
    collection_name=COLLECTION_NAME,
    data=[qvec],
    anns_field="vector",
    limit=5,
    output_fields=["id", "text", "source", "page_no"],
)

for hit in res[0]:
    score = hit_score(hit)
    ent = hit_entity(hit)

    # ent peut être dict-like ou Entity object
    get = ent.get if hasattr(ent, "get") else lambda k, d=None: getattr(ent, k, d)

    print("score:", score)
    print(get("id"), get("source"), get("page_no"))
    print((get("text") or "")[:300], "\n---\n")


score: 0.6634749174118042
data/240504434v5.pdf::s3::c3_4_14 data/240504434v5.pdf 4
We construct a high-quality and multi-source pre-training corpus consisting of 8.1T tokens. Compared with the corpus used in DeepSeek 67B (our previous release) (DeepSeek-AI, 2024), this corpus features an extended amount of data, especially Chinese data, and higher data quality. We first pretrain D 
---

score: 0.6340438723564148
data/240504434v5.pdf::s1::c1_1_6 data/240504434v5.pdf 1
## Abstract

We present DeepSeek-V2, a strong Mixture-of-Experts (MoE) language model characterized by economical training and efficient inference. It comprises 236B total parameters, of which 21B are activated for each token, and supports a context length of 128K tokens. DeepSeek-V2 adopts innovati 
---

score: 0.6225350499153137
data/240504434v5.pdf::s28::c28_19_10 data/240504434v5.pdf 19
72B Chat on both Chinese reasoning and language. Moreover, both DeepSeek-V2 Chat (SFT) and DeepSeek-V2 Chat (RL) outperform GPT-4-0613

# 2. Recherche hybride

In [ ]:
# Création de la BDD

from pymilvus import MilvusClient, DataType, Function, FunctionType

MILVUS_URI = "http://localhost:19530"
COLL = "rag_minist_int_hybrid"
EMBED_DIM = 1024

client = MilvusClient(MILVUS_URI)

if client.has_collection(COLL):
    client.drop_collection(COLL)

schema = MilvusClient.create_schema(auto_id=False, enable_dynamic_field=True)

schema.add_field(field_name="id", datatype=DataType.VARCHAR, is_primary=True, max_length=512)
schema.add_field(field_name="vector", datatype=DataType.FLOAT_VECTOR, dim=EMBED_DIM)

# texte indexable (BM25)
schema.add_field(
    field_name="text",
    datatype=DataType.VARCHAR,
    max_length=65535,
    enable_analyzer=True,
    analyzer_params={"type": "english"},  # à ajuster si tu veux un analyzer FR
    enable_match=True,
)

# sparse output du BM25
schema.add_field(field_name="sparse", datatype=DataType.SPARSE_FLOAT_VECTOR)

# Function BM25 : text -> sparse
schema.add_function(
    Function(
        name="bm25",
        function_type=FunctionType.BM25,
        input_field_names=["text"],
        output_field_names="sparse",
    )
)

index_params = MilvusClient.prepare_index_params()
index_params.add_index(
    field_name="vector",
    index_name="dense_index",
    index_type="HNSW",
    metric_type="COSINE",  # ou IP si tu normalises
    params={"M": 16, "efConstruction": 200},
)
index_params.add_index(
    field_name="sparse",
    index_name="sparse_index",
    index_type="SPARSE_INVERTED_INDEX",
    metric_type="BM25",
)

client.create_collection(
    collection_name=COLL,
    schema=schema,
    index_params=index_params,
    consistency_level="Strong",
)

client.load_collection(COLL, replica_number=1)


In [ ]:
# Insertion des chunks dans milvus

with open(JSON_PATH, "r", encoding="utf-8") as f:
    items = json.load(f)
    
batch = []
for obj in items:
    text = (obj.get("text") or "")[:MAX_TEXT]
    vec = emb_text(text)

    batch.append({
        "id": obj["id"],
        "vector": vec,
        "text": text,
        "source": obj.get("source", ""),
        "section_path": obj.get("section_path", ""),
        "section_title": obj.get("section_title", ""),
        "page_no": int(obj.get("page_no", -1)) if obj.get("page_no") is not None else -1,
        "chunk_type": (obj.get("meta") or {}).get("type", ""),
    })

    if len(batch) >= BATCH_SIZE:
        client.insert(COLL, batch)
        batch.clear()

if batch:
    client.insert(COLL, batch)


In [48]:
# Reconnexion à milvus

from pymilvus import MilvusClient
from ollama import Client as OllamaClient

MILVUS_URI = "http://localhost:19530"
COLL = "rag_minist_int_hybrid"

client = MilvusClient(MILVUS_URI)

assert client.has_collection(COLL), "La collection n'existe pas"

client.load_collection(COLL)


ollama_client = OllamaClient(host="http://localhost:11434")

def emb_text(text: str) -> list[float]:
    return ollama_client.embeddings(model="qwen3-embedding:0.6b", prompt=text)["embedding"]


In [52]:
# Requête hybride (dense + BM25) avec RRF 

from pymilvus import AnnSearchRequest, RRFRanker

def hybrid_search(query: str, top_k=5, ef=100):
    qvec = emb_text(query)

    dense_req = AnnSearchRequest(
        data=[qvec],
        anns_field="vector",
        param={"metric_type": "COSINE", "params": {"ef": ef}},
        limit=top_k,
        expr="",
    )

    sparse_req = AnnSearchRequest(
        data=[query],                # texte brut pour BM25
        anns_field="sparse",
        param={"metric_type": "BM25"},
        limit=top_k,
        expr="",
    )

    res = client.hybrid_search(
        COLL,                         # <-- positionnel
        [sparse_req, dense_req],      # <-- positionnel
        RRFRanker(k=60),                  # <-- positionnel OBLIGATOIRE
        limit=top_k,
        output_fields=["id", "text", "source", "page_no", "section_title"],
        consistency_level="Strong",
    )
    return res


res = hybrid_search("Quelle est l'idée principale de DeepSeek-V2 ?", top_k=5)

for h in res[0]:
    score = getattr(h, "distance", None) or getattr(h, "score", None)
    ent = getattr(h, "entity", None)
    get = ent.get if hasattr(ent, "get") else lambda k, d=None: getattr(ent, k, d)
    print("score:", score, "|", get("id"), "| page", get("page_no"))
    print((get("text") or "")[:250], "\n---\n")


score: 0.03201844170689583 | data/240504434v5.pdf::s3::c3_4_14 | page 4
We construct a high-quality and multi-source pre-training corpus consisting of 8.1T tokens. Compared with the corpus used in DeepSeek 67B (our previous release) (DeepSeek-AI, 2024), this corpus features an extended amount of data, especially Chinese  
---

score: 0.016393441706895828 | pdfs/template_lightOn.docx::s34::c34_16_7 | page 16
##  Installation et configuration d'un Hive Metastore

Namespace Paradigm

K8S Cluster

L'architecture Paradigm offre une flexibilité totale, pouvant fonctionner sur un seul nœud ou se déployer sur plusieurs, tout en maintenant une communication séc 
---

score: 0.016129031777381897 | data/240504434v5.pdf::s1::c1_1_6 | page 1
## Abstract

We present DeepSeek-V2, a strong Mixture-of-Experts (MoE) language model characterized by economical training and efficient inference. It comprises 236B total parameters, of which 21B are activated for each token, and supports a context  
---

sco

# 3. Evaluation

- générer un dataset d'évaluation : sur chaque page généré une question et rattaché son chunks à la question
- Rajouter du bruit dans la BDD (nouveau documents)
- Évaluer la capacité à bien retrouver le bon chunks 

In [ ]:
#!pip install "giskard[llm]"

In [ ]:
import os, giskard

os.environ["OLLAMA_API_BASE"] = "http://localhost:11434"

giskard.llm.set_llm_model(
    "ollama_chat/llama3.2:3b",   # import de mettre chat_ollama pour giskard
    api_base=os.environ["OLLAMA_API_BASE"],
)

giskard.llm.set_embedding_model(
    "ollama/qwen3-embedding:0.6b",
    api_base=os.environ["OLLAMA_API_BASE"],
)


## 3.1 evaluation dataset build

In [12]:
import pandas as pd

def milvus_to_df(client, collection_name: str, batch_size: int = 10_000):
    offset = 0
    rows = []

    # Adapte output_fields à ton schéma
    output_fields = ["id", "text", "source", "page_no", "section_title"]

    while True:
        batch = client.query(
            collection_name=collection_name,
            filter="",                 # ou expr="" selon ta version pymilvus
            output_fields=output_fields,
            limit=batch_size,
            offset=offset,
        )
        if not batch:
            break

        rows.extend(batch)
        offset += len(batch)

    return pd.DataFrame(rows)

df_kb = milvus_to_df(client, COLL)
df_kb.head()


,text,id,source,page_no,section_title
0,"## 2.2.1. Basic Architecture\n\nFor FFNs, we e...",data/240504434v5.pdf::s11::c11_9_7,data/240504434v5.pdf,9,2.2.1. Basic Architecture
1,## 2.2.2. Device-Limited Routing\n\nWe design ...,data/240504434v5.pdf::s12::c12_9_4,data/240504434v5.pdf,9,2.2.2. Device-Limited Routing
2,## 2.2.3. Auxiliary Loss for Load Balance\n\nW...,data/240504434v5.pdf::s13::c13_10_18,data/240504434v5.pdf,10,2.2.3. Auxiliary Loss for Load Balance
3,## 2.2.4. Token-Dropping Strategy\n\nWhile bal...,data/240504434v5.pdf::s14::c14_11_2,data/240504434v5.pdf,11,2.2.4. Token-Dropping Strategy
4,## 3.1.1. Data Construction\n\nWhile maintaini...,data/240504434v5.pdf::s17::c17_11_4,data/240504434v5.pdf,11,3.1.1. Data Construction


In [36]:
import numpy as np
import pandas as pd
from typing import List, Optional
from dataclasses import dataclass

from giskard.rag import KnowledgeBase, generate_testset


# ----------------------------
# ChatMessage fallback (selon ta version)
# ----------------------------
ChatMessage = None
try:
    from giskard.llm import ChatMessage  # certaines versions
except Exception:
    try:
        from giskard.llm.client import ChatMessage
    except Exception:
        try:
            from giskard.llm.client.base import ChatMessage
        except Exception:
            ChatMessage = None

if ChatMessage is None:
    @dataclass
    class ChatMessage:
        role: str
        content: str


# ----------------------------
# Wrappers Ollama (SANS LiteLLM)
# ----------------------------
class OllamaLLMClient:
    def __init__(self, ollama_client, model: str):
        self.ollama_client = ollama_client
        self.model = model

    def complete(
        self,
        messages: List[ChatMessage],
        temperature: float = 0.0,
        max_tokens: Optional[int] = None,
        caller_id: Optional[int] = None,
        seed: Optional[int] = None,
        format=None,
    ) -> ChatMessage:
        ollama_messages = [{"role": m.role, "content": m.content} for m in messages]
        options = {"temperature": temperature}
        if max_tokens is not None:
            options["num_predict"] = max_tokens

        resp = self.ollama_client.chat(
            model=self.model,
            messages=ollama_messages,
            options=options,
        )
        

        content = None
        if isinstance(resp, dict):
            content = resp.get("message", {}).get("content")
        if content is None:
            content = resp.message.content

        return ChatMessage(role="assistant", content=content)


class OllamaEmbeddingModel:
    def __init__(self, embed_fn, batch_size: int = 32):
        self.embed_fn = embed_fn
        self.batch_size = batch_size

    def embed(self, texts: List[str]):
        out = []
        for i in range(0, len(texts), self.batch_size):
            batch = texts[i:i+self.batch_size]
            out.extend([self.embed_fn(t) for t in batch])
        return np.array(out)


# ============================================================
# 1) df_kb -> colonne "document"
# ============================================================
df_kb = df_kb.copy()
df_kb["page_no"] = pd.to_numeric(df_kb.get("page_no"), errors="coerce")

df_kb["document"] = (
    "Source: " + df_kb.get("source", "").fillna("").astype(str) + "\n"
    "Page: " + df_kb["page_no"].fillna(-1).astype(int).astype(str) + "\n"
    "Section: " + df_kb.get("section_title", "").fillna("").astype(str) + "\n\n"
    + df_kb.get("text", "").fillna("").astype(str)
)

df_kb = df_kb[df_kb["document"].str.len() > 80].reset_index(drop=True)


# ============================================================
# 2) KnowledgeBase avec LLM + embeddings custom
# ============================================================
llm_client = OllamaLLMClient(ollama_client=ollama_client, model="llama3.2:3b")
embedding_model = OllamaEmbeddingModel(emb_text, batch_size=16)

knowledge_base = KnowledgeBase.from_pandas(
    df_kb,
    columns=["document"],
    chunk_size=2048,
    llm_client=llm_client,
    embedding_model=embedding_model,
)


# ============================================================
# 3) Question generators : subclass pour override la property _llm_client
# ============================================================
from giskard.rag.question_generators.simple_questions import SimpleQuestionsGenerator
from giskard.rag.question_generators.complex_questions import ComplexQuestionsGenerator

class SimpleQuestionsGeneratorOllama(SimpleQuestionsGenerator):
    @property
    def _llm_client(self):
        return llm_client

class ComplexQuestionsGeneratorOllama(ComplexQuestionsGenerator):
    @property
    def _llm_client(self):
        return llm_client

question_generators = [
    SimpleQuestionsGeneratorOllama(),
    ComplexQuestionsGeneratorOllama(),
]


# ============================================================
# 4) Generate testset
# ============================================================
testset = generate_testset(
    knowledge_base,
    num_questions=10,
    language="fr",
    agent_description=(
        "Assistant qui répond uniquement à partir des documents fournis. "
        "Il doit citer le contexte pertinent et ne pas inventer."
    ),
    question_generators=question_generators,  # ✅ force nos generators à utiliser OllamaLLMClient
)

testset.save("data/testset.jsonl")
df_test = testset.to_pandas()
df_test.head()


2026-01-14 17:40:41,566 pid:18639 MainThread giskard.rag  INFO     Finding topics in the knowledge base.


/opt/anaconda3/envs/docling_env/lib/python3.11/site-packages/umap/umap_.py:2462: UserWarning: n_neighbors is larger than the dataset size; truncating to X.shape[0] - 1
  warn(
/opt/anaconda3/envs/docling_env/lib/python3.11/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


2026-01-14 17:41:06,637 pid:18639 MainThread giskard.rag  INFO     Found 3 topics in the knowledge base.


Generating questions:   0%|          | 0/10 [00:00<?, ?it/s]

/opt/anaconda3/envs/docling_env/lib/python3.11/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 9 fields but got 5: Expected `Message` - serialized value may not be as expected [field_name='choices', input_value=Message(content='{\n    "...er_specific_fields=None), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [field_name='choices', input_value=Choices(finish_reason='st...r_specific_fields=None)), input_type=Choices])
  return self.__pydantic_serializer__.to_python(


,question,reference_answer,reference_context,conversation_history,metadata
id,,,,,
e3647764-9962-4baf-ba23-f2d15b9b51ea,Quel est le nombre de groupes nécessaires pour...,2,Document 36: Source: data/240504434v5.pdf\nPag...,[],"{'question_type': 'simple', 'seed_document_id'..."
9dca8cdf-2cf9-4f1c-91d4-25f104186ee1,Quel est le modèle de classification utilisé d...,Le modèle de classification utilisé dans la ta...,Document 18: Source: data/240504434v5.pdf\nPag...,[],"{'question_type': 'simple', 'seed_document_id'..."
047c084d-bc88-4c2a-99df-3dc4cc71c655,Quels sont les paramètres totaux du modèle Dee...,236B,Document 19: Source: data/240504434v5.pdf\nPag...,[],"{'question_type': 'simple', 'seed_document_id'..."
f9f570dd-7dea-4065-ae02-86bd02b182ee,Quel est le modèle qui a obtenu le taux de vic...,DeepSeek-V2 Chat (RL),Document 20: Source: data/240504434v5.pdf\nPag...,[],"{'question_type': 'simple', 'seed_document_id'..."
0d1c91cf-0117-4824-b201-49d5a3c01445,Quelle est la dimension du cache KV après comp...,𝑑𝑐,Document 32: Source: data/240504434v5.pdf\nPag...,[],"{'question_type': 'simple', 'seed_document_id'..."


## 3.2 metrics

In [40]:
df_test = testset.to_pandas()
print(df_test.columns)
df_test.head(2)


Index(['question', 'reference_answer', 'reference_context',
       'conversation_history', 'metadata'],
      dtype='object')


,question,reference_answer,reference_context,conversation_history,metadata
id,,,,,
e3647764-9962-4baf-ba23-f2d15b9b51ea,Quel est le nombre de groupes nécessaires pour...,2,Document 36: Source: data/240504434v5.pdf\nPag...,[],"{'question_type': 'simple', 'seed_document_id'..."
9dca8cdf-2cf9-4f1c-91d4-25f104186ee1,Quel est le modèle de classification utilisé d...,Le modèle de classification utilisé dans la ta...,Document 18: Source: data/240504434v5.pdf\nPag...,[],"{'question_type': 'simple', 'seed_document_id'..."


In [42]:
import re

def extract_source_page(context: str):
    if not isinstance(context, str):
        return None, None
    src = None
    page = None

    m = re.search(r"Source:\s*(.+)", context)
    if m:
        src = m.group(1).strip()

    m = re.search(r"Page:\s*([0-9]+)", context)
    if m:
        page = int(m.group(1))

    return src, page



def eval_one_question(question: str, ref_context: str, top_k=10):
    ref_source, ref_page = extract_source_page(ref_context)

    res = hybrid_search(question, top_k=top_k)
    hits = []

    for rank, h in enumerate(res[0], start=1):
        ent = getattr(h, "entity", None)
        get = ent.get if hasattr(ent, "get") else lambda k, d=None: getattr(ent, k, d)

        hits.append({
            "rank": rank,
            "id": get("id"),
            "source": get("source"),
            "page_no": get("page_no"),
            "score": getattr(h, "distance", None) or getattr(h, "score", None),
            "text": get("text"),
        })

    # Matching “bon chunk”
    gt_rank = None
    if ref_source is not None and ref_page is not None:
        for r in hits:
            if str(r["source"]).strip() == str(ref_source).strip() and int(r["page_no"]) == int(ref_page):
                gt_rank = r["rank"]
                break

    return gt_rank, hits


import numpy as np
import pandas as pd

def evaluate_retriever(df_test: pd.DataFrame, top_k=10, context_col="reference_context"):
    ranks = []
    rows = []

    for i, row in df_test.iterrows():
        q = row["question"]
        ref_ctx = row.get(context_col, "")

        gt_rank, hits = eval_one_question(q, ref_ctx, top_k=top_k)
        ranks.append(gt_rank if gt_rank is not None else np.inf)

        rows.append({
            "question": q,
            "gt_rank": gt_rank,
            "hit@{}".format(top_k): gt_rank is not None and gt_rank <= top_k,
            "top1_id": hits[0]["id"] if hits else None,
            "top1_source": hits[0]["source"] if hits else None,
            "top1_page": hits[0]["page_no"] if hits else None,
        })

    ranks_finite = [r for r in ranks if np.isfinite(r)]
    hit_rate = np.mean([r <= top_k for r in ranks_finite]) if ranks_finite else 0.0
    mrr = np.mean([1.0/r for r in ranks_finite]) if ranks_finite else 0.0

    return pd.DataFrame(rows), {"hit_rate": hit_rate, "mrr": mrr, "num_eval": len(df_test)}



In [ ]:
df_eval, metrics = evaluate_retriever(df_test, top_k=10, context_col="reference_context")
metrics


{'hit_rate': 1.0, 'mrr': 0.6333333333333333, 'num_eval': 10}

In [ ]:
#  # cas où le bon chunk n’est pas retrouvé
# df_eval.sort_values("gt_rank").head(10)
# df_eval[df_eval["gt_rank"].isna()].head(10) 

,question,gt_rank,hit@10,top1_id,top1_source,top1_page


## 3.3 Avec du bruit

In [51]:
import uuid
import json

MAX_TEXT = 60000
BATCH_SIZE = 256
JSON_PATH = "chunks_2.json"

def make_noise_id(prefix="noise"):
    return f"{prefix}_{uuid.uuid4().hex}"

def insert_new_chunks(items, coll=COLL, default_source="noise", chunk_type="noise"):
    batch = []
    inserted = 0

    for obj in items:
        text = (obj.get("text") or "")[:MAX_TEXT]
        if not text.strip():
            continue

        vec = emb_text(text)  # -> dim 1024 attendu

        batch.append({
            "id": obj.get("id") or make_noise_id(),   # ✅ id unique si non fourni
            "vector": vec,
            "text": text,                             # ✅ déclenche BM25 -> sparse via Function
            "source": obj.get("source", default_source),
            "section_path": obj.get("section_path", ""),
            "section_title": obj.get("section_title", ""),
            "page_no": int(obj.get("page_no", -1)) if obj.get("page_no") is not None else -1,
            "chunk_type": obj.get("chunk_type", chunk_type),
        })

        if len(batch) >= BATCH_SIZE:
            client.insert(coll, batch)
            inserted += len(batch)
            batch.clear()

    if batch:
        client.insert(coll, batch)
        inserted += len(batch)

    client.flush(coll)       
    # client.load_collection(coll)  # en général pas nécessaire si déjà load

    return inserted

with open(JSON_PATH, "r", encoding="utf-8") as f:
    noise_items = json.load(f)


n = insert_new_chunks(noise_items, coll=COLL)
print("Inserted noise chunks:", n)

Inserted noise chunks: 44


In [54]:
import re

def extract_source_page(context: str):
    if not isinstance(context, str):
        return None, None
    src = None
    page = None

    m = re.search(r"Source:\s*(.+)", context)
    if m:
        src = m.group(1).strip()

    m = re.search(r"Page:\s*([0-9]+)", context)
    if m:
        page = int(m.group(1))

    return src, page



def eval_one_question(question: str, ref_context: str, top_k=10):
    ref_source, ref_page = extract_source_page(ref_context)

    res = hybrid_search(question, top_k=top_k)
    hits = []

    for rank, h in enumerate(res[0], start=1):
        ent = getattr(h, "entity", None)
        get = ent.get if hasattr(ent, "get") else lambda k, d=None: getattr(ent, k, d)

        hits.append({
            "rank": rank,
            "id": get("id"),
            "source": get("source"),
            "page_no": get("page_no"),
            "score": getattr(h, "distance", None) or getattr(h, "score", None),
            "text": get("text"),
        })

    # Matching “bon chunk”
    gt_rank = None
    if ref_source is not None and ref_page is not None:
        for r in hits:
            if str(r["source"]).strip() == str(ref_source).strip() and int(r["page_no"]) == int(ref_page):
                gt_rank = r["rank"]
                break

    return gt_rank, hits


import numpy as np
import pandas as pd

def evaluate_retriever(df_test: pd.DataFrame, top_k=10, context_col="reference_context"):
    ranks = []
    rows = []

    for i, row in df_test.iterrows():
        q = row["question"]
        ref_ctx = row.get(context_col, "")

        gt_rank, hits = eval_one_question(q, ref_ctx, top_k=top_k)
        ranks.append(gt_rank if gt_rank is not None else np.inf)

        rows.append({
            "question": q,
            "gt_rank": gt_rank,
            "hit@{}".format(top_k): gt_rank is not None and gt_rank <= top_k,
            "top1_id": hits[0]["id"] if hits else None,
            "top1_source": hits[0]["source"] if hits else None,
            "top1_page": hits[0]["page_no"] if hits else None,
        })

    ranks_finite = [r for r in ranks if np.isfinite(r)]
    hit_rate = np.mean([r <= top_k for r in ranks_finite]) if ranks_finite else 0.0
    mrr = np.mean([1.0/r for r in ranks_finite]) if ranks_finite else 0.0

    return pd.DataFrame(rows), {"hit_rate": hit_rate, "mrr": mrr, "num_eval": len(df_test)}


df_eval, metrics = evaluate_retriever(df_test, top_k=10, context_col="reference_context")
metrics



{'hit_rate': 1.0, 'mrr': 0.4302777777777778, 'num_eval': 10}